<a href="https://colab.research.google.com/github/michaelsteven1299/proyecto_michael-/blob/main/src/01_consolidar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universidad Libre - Seccional Cali<br>Facultad de Ingeniería - Diplomado en Ciencia de Datos<br>(ↄ) Diego Fernando Marin, 2024

# 01_consolidar
Plantilla para el desarrollo del proyecto del diplomado de Ciencia de Datos, aplicando buenas prácticas.

---

Este cuaderno se enfoca en la integración de las distintas fuentes de datos en un formato cohesivo y estructurado. Aquí transformamos múltiples conjuntos de datos en una base unificada que servirá para los análisis posteriores.

**Propósito:** Crear una vista unificada y coherente de todos los datos recolectados, facilitando su posterior procesamiento y análisis.

**Tareas habituales:**
- Renombrar archivos
- Unión vertical de archivos complementarios (`union`)
- Combinar archivos (`joins`: inner, left, right, full outer)
- Estandarización inicial de formatos de columnas
- Verificación de consistencia en las uniones
- Validación de cardinalidad en las relaciones
- Gestión de duplicados producto de las uniones

In [8]:
from google.colab import drive
import os, pandas as pd

drive.mount('/content/drive')

RAW_PATH     = "/content/drive/MyDrive/proyecto_oro/data/raw/"
LANDING_PATH = "/content/drive/MyDrive/proyecto_oro/data/landing/"
os.makedirs(LANDING_PATH, exist_ok=True)

dfs = []

for archivo in os.listdir(RAW_PATH):
    if not archivo.endswith('.csv'):
        continue

    nombre = archivo.replace('.csv', '')
    df = pd.read_csv(RAW_PATH + archivo, index_col=0, parse_dates=True)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.columns = [f"{nombre}_{col}" for col in df.columns]
    dfs.append(df)
    print(f"✓ {archivo}: {df.shape[0]:,} filas × {df.shape[1]} columnas")

df_landing = pd.concat(dfs, axis=1, join='outer')
df_landing = df_landing.sort_index()

# Nos quedamos solo con días hábiles bancarios (lunes-viernes)
df_landing = df_landing[df_landing.index.dayofweek < 5]

# Política de consolidación: SIN relleno artificial (ffill).
# Usamos intersección real de fechas -> eliminamos cualquier fila que
# tenga al menos un NaN, para no distorsionar el hallazgo de
# estacionalidad por día de la semana en oro_COP.
filas_antes = len(df_landing)
df_landing = df_landing.dropna()
filas_despues = len(df_landing)

print(f"\n⚠ Filas descartadas por tener algún dato faltante: {filas_antes - filas_despues}")
print(f"   ({filas_antes:,} → {filas_despues:,} filas)")

df_landing.index.name = 'DATE'
df_landing = df_landing.reset_index()
df_landing['DATE'] = df_landing['DATE'].astype(str)

ruta_salida = LANDING_PATH + "consolidado_oro_dxy.csv"
df_landing.to_csv(ruta_salida, index=False)

print(f"\n✅ Landing listo")
print(f"📊 {df_landing.shape[0]:,} filas × {df_landing.shape[1]} columnas")
print(f"📅 {df_landing['DATE'].min()} → {df_landing['DATE'].max()}")
df_landing.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ oro_xauusd.csv: 1,378 filas × 5 columnas
✓ usd_cop.csv: 1,428 filas × 5 columnas
✓ wti_crudo.csv: 1,380 filas × 5 columnas
✓ vix.csv: 1,379 filas × 5 columnas
✓ dxy.csv: 1,380 filas × 5 columnas
✓ trm.csv: 1,299 filas × 1 columnas
✓ bono_10y.csv: 1,378 filas × 5 columnas

⚠ Filas descartadas por tener algún dato faltante: 419
   (1,431 → 1,012 filas)

✅ Landing listo
📊 1,012 filas × 32 columnas
📅 2021-01-05 → 2026-06-26


,DATE,oro_xauusd_Close,oro_xauusd_High,oro_xauusd_Low,oro_xauusd_Open,oro_xauusd_Volume,usd_cop_Close,usd_cop_High,usd_cop_Low,usd_cop_Open,...,dxy_High,dxy_Low,dxy_Open,dxy_Volume,trm_trm,bono_10y_Close,bono_10y_High,bono_10y_Low,bono_10y_Open,bono_10y_Volume
0,2021-01-05,182.869995,183.210007,181.820007,182.869995,12718800.0,3447.750000,3467.550049,3428.679932,3447.750000,...,89.900002,89.430000,89.900002,0.0,3420.78,0.955,0.963,0.927,0.937,0.0
1,2021-01-06,179.899994,181.580002,178.240005,181.490005,18453500.0,3442.250000,3442.250000,3401.046387,3442.250000,...,89.800003,89.209999,89.480003,0.0,3450.74,1.042,1.054,1.000,1.000,0.0
2,2021-01-07,179.479996,179.919998,178.839996,179.690002,7110200.0,3412.800049,3477.570068,3374.599121,3412.800049,...,89.970001,89.320000,89.320000,0.0,3428.04,1.071,1.088,1.054,1.056,0.0
3,2021-01-08,173.339996,176.990005,171.479996,176.830002,24399900.0,3488.239990,3497.129883,3449.190674,3488.239990,...,90.250000,89.660004,89.830002,0.0,3459.39,1.105,1.126,1.075,1.088,0.0
4,2021-01-13,173.369995,174.460007,173.050003,173.679993,14110300.0,3475.250000,3487.580078,3467.590088,3475.250000,...,90.449997,89.919998,90.000000,0.0,3487.65,1.088,1.133,1.073,1.129,0.0


In [7]:
import os
print(os.listdir(RAW_PATH))

['oro_xauusd.csv', 'usd_cop.csv', 'wti_crudo.csv', 'vix.csv', 'dxy.csv', 'trm.csv', 'bono_10y.csv']
